# LIMPIEZA DE LOS DATOS DE TMDB

Se obtuvo un archivo JSON por cada película almacenada en movies.csv (MovieLens), mediante llamadas a la API de TMDB. En este cuaderno se describe el proceso de consolidación de esos archivos JSON, selección de los campos de interés y transformación a un único archivo TMDB_clean.parquet.

**Entrada**: archivos JSON individuales en data/01_raw/TMDB \
**Objetivos**: lectura, consolidación, selección de campos de interés y transformación \
**Salida**: TMDB_clean.parquet

A diferencia de los demás cuadernos de limpieza, aquí no se realiza una validación explícita de nulos o duplicados sobre los campos de TMDB, ya que el objetivo principal es consolidar la información y quedarse con las columnas relevantes para el proyecto.

In [1]:
import os
import json
import pandas as pd

Se leen todos los archivos JSON almacenados en data/01_raw/TMDB, uno por película, y se construye un único DataFrame a partir de ellos.

In [2]:
TMDB = "../data/01_raw/TMDB"
datos = []
for archivo in os.listdir(TMDB):
    ruta = os.path.join(TMDB, archivo)
    with open(ruta, 'r') as f:
        datos.append(json.load(f))
datos
df = pd.DataFrame(datos)
df.head()

,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,revenue,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,/cXQH2u7wUIX1eoIdEj51kHXoWhX.jpg,None,1350000,"[{'id': 35, 'name': 'Comedy'}, {'id': 80, 'nam...",http://www.universalstudiosentertainment.com/l...,100,tt0120735,[GB],en,...,28356188,105,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,A Disgrace to Criminals Everywhere.,"Lock, Stock and Two Smoking Barrels",False,8.096,7207
1,False,/ke8MuTHVveodT0YQxV1TcTh4BXd.jpg,None,24900000,"[{'id': 35, 'name': 'Comedy'}, {'id': 878, 'na...",,10001,tt0096486,"[AU, US]",en,...,5000000,91,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,The Incredible Untold Story of the Greatest Mi...,Young Einstein,False,5.300,172
2,False,/ejnFWUUTYbeQ70WqYFsx8iaDadW.jpg,None,0,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...",,10002,tt0091538,[GB],en,...,5794184,105,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Sometimes love is a strange and wicked game.,Mona Lisa,False,6.900,292
3,False,/3cdfnihGSrMiQWzmVPaEs3p2Mp1.jpg,None,68000000,"[{'id': 53, 'name': 'Thriller'}, {'id': 28, 'n...",,10003,tt0120053,[US],en,...,118100000,116,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Never reveal your name. Never turn your back. ...,The Saint,False,6.110,1245
4,False,/436HdmXTod7X0aafPPoIZyaecgh.jpg,"{'id': 96665, 'name': 'Dumb and Dumber Collect...",40000000,"[{'id': 35, 'name': 'Comedy'}]",,100042,tt2096672,[US],en,...,169837010,110,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,The average person uses 10% of their brain cap...,Dumb and Dumber To,False,5.600,3421


In [3]:
df.info() #dimensiones coinciden con el número de archivos

<class 'pandas.DataFrame'>
RangeIndex: 9620 entries, 0 to 9619
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  9620 non-null   bool   
 1   backdrop_path          9537 non-null   str    
 2   belongs_to_collection  2115 non-null   object 
 3   budget                 9620 non-null   int64  
 4   genres                 9620 non-null   object 
 5   homepage               9620 non-null   str    
 6   id                     9620 non-null   int64  
 7   imdb_id                9620 non-null   str    
 8   origin_country         9620 non-null   object 
 9   original_language      9620 non-null   str    
 10  original_title         9620 non-null   str    
 11  overview               9620 non-null   str    
 12  popularity             9620 non-null   float64
 13  poster_path            9616 non-null   str    
 14  production_companies   9620 non-null   object 
 15  production_coun

El número de filas coincide con el número de archivos JSON leídos. Algunas columnas tienen valores no nulos por debajo del total, como `belongs_to_collection` (solo presente cuando la película pertenece a una saga), `poster_path`, `backdrop_path` o `softcore`; se trata de campos que TMDB no siempre proporciona, no de errores de lectura.

Se seleccionan los campos de interés para el proyecto: `id` (identificador de TMDB, equivalente a `tmdbId`), `title`, `genres`, `popularity`, `overview`, `tagline`, `vote_average`, `vote_count`, `runtime`, `budget`, `revenue` y `release_date`. El resto de columnas del JSON original no se utilizan y se descartan. Además, se ordena el resultado por `id`.

In [4]:
campos = ['id','title','genres','popularity','overview','tagline','vote_average','vote_count', 'runtime','budget','revenue', 'release_date']
df2 = df[campos].sort_values('id')
df2.head()

,id,title,genres,popularity,overview,tagline,vote_average,vote_count,runtime,budget,revenue,release_date
3561,2,Ariel,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",1.3807,A Finnish man goes to the city to find a job a...,,7.121,375,73,0,0,1988-10-21
7372,5,Four Rooms,"[{'id': 35, 'name': 'Comedy'}]",3.6807,It's Ted the Bellhop's first night on the job....,Twelve outrageous guests. Four scandalous requ...,5.904,2847,98,4000000,4257354,1995-12-09
7802,6,Judgment Night,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",1.8049,"Four young friends, while taking a shortcut en...",Don't move. Don't whisper. Don't even breathe.,6.463,377,109,21000000,12136938,1993-10-15
693,11,Star Wars,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",27.0355,Princess Leia is captured and held hostage by ...,"A long time ago in a galaxy far, far away...",8.205,22361,121,11000000,775398007,1977-05-25
1331,12,Finding Nemo,"[{'id': 16, 'name': 'Animation'}, {'id': 10751...",19.5501,"Nemo, an adventurous young clownfish, is unexp...",There are 3.7 trillion fish in the ocean. They...,7.819,20567,100,94000000,940335536,2003-05-30


La columna `genres` llega desde TMDB como una lista de diccionarios con el identificador y el nombre de cada género. Se simplifica quedándose solo con la lista de nombres de género, más manejable para el resto del proyecto.

In [5]:
df2['genres'] = df2['genres'].apply(lambda lista: [g['name'] for g in lista ])
df2.head()

,id,title,genres,popularity,overview,tagline,vote_average,vote_count,runtime,budget,revenue,release_date
3561,2,Ariel,"[Comedy, Drama, Romance, Crime]",1.3807,A Finnish man goes to the city to find a job a...,,7.121,375,73,0,0,1988-10-21
7372,5,Four Rooms,[Comedy],3.6807,It's Ted the Bellhop's first night on the job....,Twelve outrageous guests. Four scandalous requ...,5.904,2847,98,4000000,4257354,1995-12-09
7802,6,Judgment Night,"[Action, Crime, Thriller]",1.8049,"Four young friends, while taking a shortcut en...",Don't move. Don't whisper. Don't even breathe.,6.463,377,109,21000000,12136938,1993-10-15
693,11,Star Wars,"[Adventure, Action, Science Fiction]",27.0355,Princess Leia is captured and held hostage by ...,"A long time ago in a galaxy far, far away...",8.205,22361,121,11000000,775398007,1977-05-25
1331,12,Finding Nemo,"[Animation, Family, Adventure]",19.5501,"Nemo, an adventurous young clownfish, is unexp...",There are 3.7 trillion fish in the ocean. They...,7.819,20567,100,94000000,940335536,2003-05-30


## Transformación final

Se guarda el resultado, con los campos seleccionados y los géneros simplificados, en `TMDB_clean.parquet`.

In [6]:
df2.to_parquet('../data/02_processed/TMDB_clean.parquet',index=False)